# Merge all fine labeled cell states into a unified atlas

**Goal.** Build one cell-state column (`cell_state_x`) on the full snRNA-seq atlas by overlaying
the fine labels from each compartment object onto the broad atlas labels, then export the inputs
CellPhoneDB needs.

**Logic.** Start from the broad label (`cell_type_new`) → for each compartment object, replace the
label of matching barcodes with that object's fine label (malignant states, macrophage / myeloid
states, lymphoid states). Cells not in any compartment object keep their broad label.

**Output.**
- `new_for_Cellphone_cell_state.h5ad`: atlas with `cell_state_x` (log-normalised `X`).
- `cpdb_meta.tsv`: `barcode_sample`, `cell_type` table for CellPhoneDB.

In [ ]:
# !pip install --quiet scanpy
# %cd <project>/cpdb

In [ ]:
import os

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

## Config

In [ ]:
atlas_path = "scVI_integrated_adata_clean_latent20_major_cell_type_inferCNV_add_copyKAT_score_ctype_new.h5ad"

base_label_col = "cell_type_new"        # broad label kept where no fine label exists
new_label_col = "cell_state_x"

# compartment object -> (fine label column, broad atlas label(s) it is expected to refine)
# applied in order; a later source overwrites an earlier one for shared barcodes
sources = {
    "../h5ad/tumour/pulmonary_epi_Lin/sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join.h5ad":
        ("final_state", ["Malignant", "Alveolar cell type I", "Alveolar cell type II"]),
    "../h5ad/myeloid/sub_Myeloid_raw_scvi_combine_annotated.h5ad": ("cell_state", ["Myeloid"]),
    "../h5ad/lymphoid/sub_Lymphoid_raw_scvi_combine_annotated.h5ad": ("cell_state", ["Lymphoid"]),
}
# Option 1: use "cell_state_v2" for myeloid / lymphoid instead of "cell_state"

drop_labels = []                        # e.g. ["unassigned", "Malignant"] to exclude from CellPhoneDB
expr_layer = "logcounts"                # CellPhoneDB input: log-normalised, not raw counts

output_h5ad = "new_for_Cellphone_cell_state_fixed.h5ad" #Final object {new_for_Cellphone_cell_state_fixed.h5ad}
output_meta = "cpdb_meta.tsv"

## 1. Load atlas

Compartment objects are opened in backed mode: only their `obs` is needed.

In [ ]:
adata = sc.read_h5ad(atlas_path)
assert base_label_col in adata.obs, f"'{base_label_col}' missing in atlas"
labels = adata.obs[base_label_col].astype(str).copy()
adata

## 2. Overlay fine labels

Per source: how many barcodes match the atlas, how many are missing from it, and which broad
labels they replace (cells outside `expected` broad labels flag a barcode or annotation mismatch).

In [ ]:
report, seen = [], pd.Index([])
for path, (col, expected) in sources.items():
    obs = ad.read_h5ad(path, backed="r").obs
    assert col in obs, f"'{col}' missing in {path}"
    shared = adata.obs_names.intersection(obs.index)
    broad = labels.loc[shared]
    report.append({
        "source": os.path.basename(path), "label_col": col, "n_source": len(obs), "n_matched": len(shared),
        "n_not_in_atlas": len(obs) - len(shared), "n_overwrites_previous": len(shared.intersection(seen)),
        "n_outside_expected": int((~adata.obs.loc[shared, base_label_col].astype(str).isin(expected)).sum()),
    })
    labels.loc[shared] = obs.loc[shared, col].astype(str).values
    seen = seen.union(shared)

adata.obs[new_label_col] = pd.Categorical(labels)
pd.DataFrame(report).set_index("source")

In [ ]:
# which broad labels were refined, and which were left as is
pd.crosstab(adata.obs[new_label_col], adata.obs[base_label_col]).loc[
    lambda d: d.sum(axis=1) > 0
]

In [ ]:
counts = adata.obs[new_label_col].value_counts()
unrefined = [c for spec in sources.values() for c in spec[1] if c in counts.index]
print("broad labels still present after overlay:", {c: int(counts[c]) for c in unrefined})
counts

In [ ]:
sc.pl.umap(adata, color=[base_label_col, new_label_col], ncols=1, frameon=False)

## 3. Export for CellPhoneDB

`X` is set to the log-normalised layer; labels in `drop_labels` are removed.

In [ ]:
if drop_labels:
    adata = adata[~adata.obs[new_label_col].isin(drop_labels)].copy()
    adata.obs[new_label_col] = adata.obs[new_label_col].cat.remove_unused_categories()

adata.X = adata.layers[expr_layer].copy()
vals = adata.X[:1000].data if hasattr(adata.X, "data") else np.asarray(adata.X[:1000]).ravel()
assert not np.allclose(vals, np.round(vals)), f"'{expr_layer}' looks like raw counts"

adata.write_h5ad(output_h5ad)
meta = pd.DataFrame({"barcode_sample": adata.obs_names, "cell_type": adata.obs[new_label_col].astype(str).values})
meta.to_csv(output_meta, sep="\t", index=False)
print(f"{adata.n_obs} cells, {meta['cell_type'].nunique()} cell types -> {output_h5ad}, {output_meta}")